# Method comparison on the hold-out S21, two example users and list properties

All lists are built from history weeks 11 to 20 and compared with the target of weeks 21 to 30. The two example users are fixed by a rule before any list is looked at: among test users with a target on S21, A is the user closest to the 90th percentile of labelled viewing hours in weeks 11 to 20 and B the user closest to the 10th percentile among those with at least one airing watched for 0.8 of its length.

In [1]:
# Environment, data, snapshot S21 and the lists of every method
import json
import os
import platform
import sys
import time

import numpy as np
import pandas as pd

ROOT = os.path.abspath('..')
sys.path.insert(0, os.path.join(ROOT, 'src'))
from sweettv import baselines, cf, data, recs, snapshot, target  # noqa: E402

RAW, CACHE = os.path.join(ROOT, 'data', 'raw'), os.path.join(ROOT, 'data', 'cache')
SUB_DIR = os.path.join(ROOT, 'submissions')
pd.set_option('display.width', 250)
pd.set_option('display.max_columns', 40)
pd.set_option('display.max_colwidth', 60)
print(platform.platform(), '| python', platform.python_version(), '| numpy', np.__version__, '| pandas', pd.__version__)
T0 = time.time()
views = data.load_views(RAW, CACHE)
sched = data.load_schedule(RAW, CACHE)
test_users = pd.Index(data.load_submission(RAW).user_id)
SETTINGS = json.load(open(os.path.join(ROOT, 'modelling', 'cf_results.json')))['settings']
BASE = json.load(open(os.path.join(ROOT, 'modelling', 'baselines_results.json')))['selected']
lab = sched[sched.labelled]
prog = lab.groupby('tv_show_id').agg(title=('tv_show_title', 'first'), category=('tv_show_category', 'first'),
                                     genre_1=('tv_show_genre_1', 'first'), genre_2=('tv_show_genre_2', 'first'),
                                     genre_3=('tv_show_genre_3', 'first'), year=('year_of_production', 'first'),
                                     director=('director', 'first'), actors=('actors', 'first'))
s = snapshot.build(views, sched, 21, 30)
truth = s.target
users = truth.index
fb = baselines.popularity_ranking(s)
m = cf.Interactions.from_snapshot(s)
als_kw = dict(SETTINGS['als'])
als_kw['factors'] = int(als_kw['factors'])
empty = pd.DataFrame({'user_id': pd.Series(dtype='uint64'), 'tv_show_id': pd.Series(dtype='int64'), 'score': pd.Series(dtype=float)})
score_tables = {
    'Popularity': empty,
    'Habit x schedule': baselines.habit_schedule_scores(s, BASE['half_life_weeks'], BASE['prior']),
    'ALS': cf.als(m, **als_kw).long(s.pool, 50),
    'PPR': cf.ppr(m, **SETTINGS['ppr']).long(s.pool, 50),
    'Association rules': cf.association_rules(m, **SETTINGS['rules']).long(s.pool, 50),
    'Content-based': cf.content(m, prog, sorted(s.pool)).long(s.pool, 50),
}
lists = {name: recs.top5(sc, users, fb, s.pool) for name, sc in score_tables.items()}
lists['Hybrid (LightGBM re-ranker)'] = recs.read_submission(os.path.join(ROOT, 'modelling', 's21_ranker_v3_top5.csv'))
ap = pd.DataFrame({name: [target.apk(truth[u], lst[u]) for u in users] for name, lst in lists.items()}, index=users)
print('MAP@5 on S21:', ap.mean().round(4).to_dict())

Windows-11-10.0.26200-SP0 | python 3.12.14 | numpy 2.5.4 | pandas 3.0.6


MAP@5 on S21: {'Popularity': 0.2341, 'Habit x schedule': 0.3366, 'ALS': 0.2832, 'PPR': 0.3184, 'Association rules': 0.2545, 'Content-based': 0.0854, 'Hybrid (LightGBM re-ranker)': 0.3595}


In [2]:
# Paired bootstrap over users: difference of MAP@5 against the hybrid, 95% intervals
rng = np.random.default_rng(0)
idx = rng.integers(0, len(ap), size=(2000, len(ap)))
rows = {}
for name in ap.columns:
    d = (ap['Hybrid (LightGBM re-ranker)'] - ap[name]).to_numpy()
    boots = d[idx].mean(axis=1)
    rows[name] = {'MAP@5': ap[name].mean(), 'hybrid minus method': d.mean(), 'ci low': np.quantile(boots, 0.025),
                  'ci high': np.quantile(boots, 0.975), 'users where hybrid is higher': int((d > 0).sum()),
                  'users where hybrid is lower': int((d < 0).sum())}
display(pd.DataFrame(rows).T.round(4))

,MAP@5,hybrid minus method,ci low,ci high,users where hybrid is higher,users where hybrid is lower
Popularity,0.2341,0.1254,0.1192,0.1319,2396.0,730.0
Habit x schedule,0.3366,0.0229,0.0197,0.0261,1452.0,849.0
ALS,0.2832,0.0763,0.0717,0.0812,2139.0,703.0
PPR,0.3184,0.0411,0.0378,0.0445,1726.0,711.0
Association rules,0.2545,0.1050,0.0997,0.1108,2302.0,714.0
Content-based,0.0854,0.2741,0.2669,0.2818,3334.0,103.0
Hybrid (LightGBM re-ranker),0.3595,0.0000,0.0000,0.0000,0.0,0.0


In [3]:
# MAP@5 by user activity tercile (labelled viewing hours in weeks 11 to 20) and for test users
hours = s.history.groupby('user_id').sec.sum().reindex(users).fillna(0) / 3600
terc = pd.qcut(hours.rank(method='first'), 3, labels=['light', 'medium', 'heavy'])
seg = ap.groupby(terc, observed=True).mean().T
seg['test users'] = ap[ap.index.isin(test_users)].mean()
display(seg.round(4))

sec,light,medium,heavy,test users
Popularity,0.1480,0.2276,0.3268,0.2592
Habit x schedule,0.2117,0.3338,0.4644,0.3770
ALS,0.1902,0.2888,0.3706,0.3064
PPR,0.1929,0.3170,0.4454,0.3572
Association rules,0.1646,0.2518,0.3471,0.2793
Content-based,0.0792,0.0894,0.0875,0.0899
Hybrid (LightGBM re-ranker),0.2399,0.3572,0.4815,0.3960


In [4]:
# List properties: familiar against new programmes, popularity, coverage and overlap with the popularity list
def pairs_of(df):
    # to_numpy() on uint64 and int64 columns would upcast to float64 and corrupt the 20-digit user ids
    return set(zip(df.user_id.tolist(), df.tv_show_id.tolist()))


q_pairs = pairs_of(target.frequency(s.history))
t_pairs = pairs_of(s.history[['user_id', 'tv_show_id']].drop_duplicates())
pop_share = target.top_k(target.frequency(s.history), strict=False).explode().astype('int64').value_counts() / s.history.user_id.nunique()
pop5 = set(fb[:5])
props = {}
for name, lst in lists.items():
    flat = pd.DataFrame([(int(u), int(p)) for u in users for p in lst[u]], columns=['user_id', 'tv_show_id']).astype({'user_id': 'uint64'})
    pair = list(zip(flat.user_id.tolist(), flat.tv_show_id.tolist()))
    flat['watched'] = [x in q_pairs for x in pair]
    flat['touched'] = [x in t_pairs for x in pair]
    flat['hit'] = [p in set(truth[u]) for u, p in pair]
    new = ~flat.touched
    props[name] = {'share watched at 0.8 before': flat.watched.mean(), 'share only touched before': (flat.touched & ~flat.watched).mean(),
                   'share new to the user': new.mean(), 'precision on familiar': flat.hit[flat.touched].mean(),
                   'precision on new': flat.hit[new].mean() if new.any() else np.nan,
                   'distinct programmes': flat.tv_show_id.nunique(), 'catalogue coverage of the pool': flat.tv_show_id.nunique() / len(s.pool),
                   'mean popularity (share of users with it in their top 5)': flat.tv_show_id.map(pop_share).fillna(0).mean(),
                   'mean shared with the popularity top 5': flat.tv_show_id.isin(pop5).groupby(flat.user_id).sum().mean(),
                   'category share Инфо': flat.tv_show_id.map(prog.category).eq('Инфо').mean(),
                   'category share Фильмы': flat.tv_show_id.map(prog.category).eq('Фильмы').mean()}
props_df = pd.DataFrame(props).T
display(props_df.round(4))
tflat = truth.explode().astype('int64')
print('actual targets: share watched at 0.8 before', round(np.mean([(int(u), int(p)) in q_pairs for u, p in tflat.items()]), 4),
      '| share only touched before', round(np.mean([(int(u), int(p)) in t_pairs and (int(u), int(p)) not in q_pairs for u, p in tflat.items()]), 4),
      '| share new to the user', round(np.mean([(int(u), int(p)) not in t_pairs for u, p in tflat.items()]), 4))

,share watched at 0.8 before,share only touched before,share new to the user,precision on familiar,precision on new,distinct programmes,catalogue coverage of the pool,mean popularity (share of users with it in their top 5),mean shared with the popularity top 5,category share Инфо,category share Фильмы
Popularity,0.5973,0.2174,0.1854,0.3581,0.0680,5.0,0.0008,0.3368,5.0000,0.8000,0.0000
Habit x schedule,0.9577,0.0158,0.0265,0.3972,0.1573,482.0,0.0756,0.2197,2.4318,0.4554,0.0306
ALS,0.8677,0.0645,0.0678,0.3668,0.0930,78.0,0.0122,0.2680,2.8953,0.5400,0.0031
PPR,0.9648,0.0116,0.0237,0.3789,0.1572,730.0,0.1145,0.2226,2.3543,0.4379,0.0582
Association rules,0.6991,0.1793,0.1216,0.3522,0.0649,35.0,0.0055,0.3127,3.8504,0.7012,0.0000
Content-based,0.3525,0.1346,0.5129,0.3141,0.0100,576.0,0.0903,0.1120,1.1978,0.5301,0.0477
Hybrid (LightGBM re-ranker),0.8327,0.1212,0.0462,0.4297,0.1387,176.0,0.0276,0.2461,2.7610,0.5032,0.0043


actual targets: share watched at 0.8 before 0.5088 | share only touched before 0.1448 | share new to the user 0.3464


In [5]:
# The two example users, chosen by the rule in the header
cand = users[users.isin(test_users)]
h_hours = hours.reindex(cand)
n_q = target.frequency(s.history).groupby('user_id').size().reindex(cand).fillna(0)


def closest(series, value):
    d = (series - value).abs()
    return d[d == d.min()].index.min()


user_a = closest(h_hours, h_hours.quantile(0.9))
light = h_hours[n_q > 0]
user_b = closest(light, light.quantile(0.1))
for label, u in [('A (heavy)', user_a), ('B (light)', user_b)]:
    hq = target.frequency(s.history)
    hq = hq[hq.user_id == u].sort_values('freq', ascending=False)
    print(f'user {label}: {u} | labelled hours in weeks 11 to 20: {hours[u]:.1f} | programmes watched at 0.8: {len(hq)} | '
          f'AP@5 per method: {ap.loc[u].round(3).to_dict()}')
    display(hq.head(8).assign(title=hq.tv_show_id.map(prog.title), category=hq.tv_show_id.map(prog.category))
            [['tv_show_id', 'title', 'category', 'freq']].rename(columns={'freq': 'airings watched at 0.8, weeks 11 to 20'}))
    display(pd.DataFrame({'tv_show_id': truth[u], 'title': [prog.title[p] for p in truth[u]],
                          'category': [prog.category[p] for p in truth[u]]}).rename_axis(f'actual top 5 of user {label}, weeks 21 to 30'))


def show_lists(u):
    rows_ = []
    for name, lst in lists.items():
        for rank, p in enumerate(lst[u], start=1):
            key = (int(u), int(p))
            rows_.append({'method': name, 'rank': rank, 'tv_show_id': p, 'title': prog.title.get(p), 'category': prog.category.get(p),
                          'seen before': 'watched' if key in q_pairs else ('touched' if key in t_pairs else 'new'),
                          'in actual top 5': p in set(truth[u])})
    return pd.DataFrame(rows_).set_index(['method', 'rank'])


for label, u in [('A', user_a), ('B', user_b)]:
    print(f'Top 5 of every method for user {label} on S21')
    display(show_lists(u))

user A (heavy): 3525695269081370558 | labelled hours in weeks 11 to 20: 349.5 | programmes watched at 0.8: 47 | AP@5 per method: {'Popularity': 0.287, 'Habit x schedule': 0.76, 'ALS': 0.167, 'PPR': 0.643, 'Association rules': 0.287, 'Content-based': 0.0, 'Hybrid (LightGBM re-ranker)': 0.71}


,tv_show_id,title,category,"airings watched at 0.8, weeks 11 to 20"
20398,12001682,М/с 'Губка Боб Квадратные Штаны'.,Детям,32
20400,12001734,"М/с 'Волшебные покровители', 154 с. 'Хочу это вырезать /...",Детям,16
20386,2400480,'ТСН. Неделя'.,Инфо,10
20404,12002856,М/с 'Элвин и бурундуки'.,Детям,9
20387,2400508,Сегодня. Итоги с Олегом Панютой.,Инфо,8
20385,2400467,'Викна-Новости'.,Инфо,7
20379,400357,"'Холостяк', 10 сезон, 1 эп., 1 ч.",Развлечения,5
20390,6500479,"'Мир наизнанку. Бразилия', 36 эп.",Познавательное,4


,tv_show_id,title,category
"actual top 5 of user A (heavy), weeks 21 to 30",,,
0,12001682,М/с 'Губка Боб Квадратные Штаны'.,Детям
1,12002856,М/с 'Элвин и бурундуки'.,Детям
2,12001734,"М/с 'Волшебные покровители', 154 с. 'Хочу это вырезать /...",Детям
3,2400508,Сегодня. Итоги с Олегом Панютой.,Инфо
4,2400467,'Викна-Новости'.,Инфо


user B (light): 14049050536217794294 | labelled hours in weeks 11 to 20: 59.2 | programmes watched at 0.8: 14 | AP@5 per method: {'Popularity': 0.0, 'Habit x schedule': 0.0, 'ALS': 0.0, 'PPR': 0.0, 'Association rules': 0.0, 'Content-based': 0.0, 'Hybrid (LightGBM re-ranker)': 0.0}


,tv_show_id,title,category,"airings watched at 0.8, weeks 11 to 20"
87746,2500413,Гражданская оборона.,Инфо,2
87738,200337,'Утро с 'Интером'.,Развлечения,1
87740,400335,"'Хата на тата', 7 сезон, 1 эп.",Развлечения,1
87741,400361,"'МастерШеф', 7 сезон, 17 эп.",Познавательное,1
87742,1000381,"Т/с 'CSI: Место преступления', 7 сезон, 6 и 7 с.",Сериалы,1
87739,240081,Факты. День.,Инфо,1
87743,1000723,"Т/с 'Охотники за древностями', 3 сезон, 14 и 15 с.",Сериалы,1
87744,1100411,Премьера. 'Вещдок. Особый случай. Око за око'.,Познавательное,1


,tv_show_id,title,category
"actual top 5 of user B (light), weeks 21 to 30",,,
0,90082736,Х/ф 'Человек-паук. Вдали от дома'.,Фильмы
1,90072084,Х/ф 'РЭД'.,Фильмы
2,90057266,Х/ф 'Терминатор 3: Восстание машин'.,Фильмы
3,1000381,"Т/с 'CSI: Место преступления', 7 сезон, 6 и 7 с.",Сериалы
4,90055475,Х/ф 'Парк Юрского периода 3'.,Фильмы


Top 5 of every method for user A on S21


tv_show_id                                                        title     category seen before  in actual top 5
method                      rank                                                                                                                   
Popularity                  1        2400480                                               'ТСН. Неделя'.         Инфо     watched            False
                            2         240081                                                 Факты. День.         Инфо     watched            False
                            3        2400467                                             'Викна-Новости'.         Инфо     watched             True
                            4        2400508                             Сегодня. Итоги с Олегом Панютой.         Инфо     watched             True
                            5       12001682                            М/с 'Губка Боб Квадратные Штаны'.        Детям     watched             True
Habit x schedule            1       12001682                            М/с 'Губка Боб Квадратные Штаны'.        Детям     watched             True
                            2       12002856                                     М/с 'Элвин и бурундуки'.        Детям     watched             True
                            3       12001734  М/с 'Волшебные покровители', 154 с. 'Хочу это вырезать /...        Детям     watched             True
                            4        2400480                                               'ТСН. Неделя'.         Инфо     watched            False
                            5        2400508                             Сегодня. Итоги с Олегом Панютой.         Инфо     watched             True
ALS                         1        2400480                                               'ТСН. Неделя'.         Инфо     watched            False
                            2         240081                                                 Факты. День.         Инфо     watched            False
                            3       12001682                            М/с 'Губка Боб Квадратные Штаны'.        Детям     watched             True
                            4        2400467                                             'Викна-Новости'.         Инфо     watched             True
                            5          20088    'Завтрак с 1+1'. Информационно-развлекательная программа.  Развлечения     watched            False
PPR                         1       12001682                            М/с 'Губка Боб Квадратные Штаны'.        Детям     watched             True
                            2        2400480                                               'ТСН. Неделя'.         Инфо     watched            False
                            3       12001734  М/с 'Волшебные покровители', 154 с. 'Хочу это вырезать /...        Детям     watched             True
                            4        2400508                             Сегодня. Итоги с Олегом Панютой.         Инфо     watched             True
                            5        2400467                                             'Викна-Новости'.         Инфо     watched             True
Association rules           1        2400480                                               'ТСН. Неделя'.         Инфо     watched            False
                            2         240081                                                 Факты. День.         Инфо     watched            False
                            3        2400467                                             'Викна-Новости'.         Инфо     watched             True
                            4        2400508                             Сегодня. Итоги с Олегом Панютой.         Инфо     watched             True
                            5       12001682                            М/с 'Губка Боб Квадратные Штаны'.        Детям     watched             True
Content-based               1        5

Top 5 of every method for user B on S21


tv_show_id                                                      title        category seen before  in actual top 5
method                      rank                                                                                                                    
Popularity                  1        2400480                                             'ТСН. Неделя'.            Инфо     touched            False
                            2         240081                                               Факты. День.            Инфо     watched            False
                            3        2400467                                           'Викна-Новости'.            Инфо     watched            False
                            4        2400508                           Сегодня. Итоги с Олегом Панютой.            Инфо     touched            False
                            5       12001682                          М/с 'Губка Боб Квадратные Штаны'.           Детям     touched            False
Habit x schedule            1        2500413                                       Гражданская оборона.            Инфо     watched            False
                            2        2400467                                           'Викна-Новости'.            Инфо     watched            False
                            3         240081                                               Факты. День.            Инфо     watched            False
                            4       10003012                                    Т/с 'Однажды в Одессе'.         Сериалы     watched            False
                            5        1100411             Премьера. 'Вещдок. Особый случай. Око за око'.  Познавательное     watched            False
ALS                         1         240081                                               Факты. День.            Инфо     watched            False
                            2        2400467                                           'Викна-Новости'.            Инфо     watched            False
                            3        2500413                                       Гражданская оборона.            Инфо     watched            False
                            4         400361                               'МастерШеф', 7 сезон, 17 эп.  Познавательное     watched            False
                            5        2400480                                             'ТСН. Неделя'.            Инфо     touched            False
PPR                         1        2500413                                       Гражданская оборона.            Инфо     watched            False
                            2        2400467                                           'Викна-Новости'.            Инфо     watched            False
                            3         400361                               'МастерШеф', 7 сезон, 17 эп.  Познавательное     watched            False
                            4         240081                                               Факты. День.            Инфо     watched            False
                            5         400335                             'Хата на тата', 7 сезон, 1 эп.     Развлечения     watched            False
Association rules           1         240081                                               Факты. День.            Инфо     watched            False
                            2        2400480                                             'ТСН. Неделя'.            Инфо     touched            False
                            3        2400467                                           'Викна-Новости'.            Инфо     watched            False
                            4        2400508                           Сегодня. Итоги с Олегом Панютой.            Инфо     touched            False
                            5          20088  'Завтрак с 1+1'. Информационно-развлекательная программа.     Развлечения         new            False
Content-bas

In [6]:
# The same two users in the test window: each method's submitted five programmes (no ground truth is available)
sub_files = {'Popularity': 'popularity.csv', 'Habit x schedule': 'habit_schedule.csv', 'ALS': 'als.csv', 'PPR': 'ppr.csv',
             'Association rules': 'rules.csv', 'Content-based': 'content.csv', 'Hybrid (LightGBM re-ranker)': 'ranker_v3.csv'}
subs = {name: recs.read_submission(os.path.join(SUB_DIR, f)) for name, f in sub_files.items()}
for label, u in [('A', user_a), ('B', user_b)]:
    rows_ = [{'method': name, 'rank': k + 1, 'tv_show_id': p, 'title': prog.title.get(p), 'category': prog.category.get(p)}
             for name, sub in subs.items() for k, p in enumerate(sub[u])]
    print(f'Submitted five programmes for user {label} ({u}), weeks 31 to 42')
    display(pd.DataFrame(rows_).set_index(['method', 'rank']))


def example(u):
    hq = target.frequency(s.history)
    hq = hq[hq.user_id == u].sort_values('freq', ascending=False)
    seen = lambda p: 'watched' if (int(u), int(p)) in q_pairs else ('touched' if (int(u), int(p)) in t_pairs else 'new')  # noqa: E731
    return {'user_id': int(u), 'hours_w11_20': float(hours[u]), 'programmes_watched_w11_20': int(len(hq)),
            'history_top': [[int(p), int(c)] for p, c in zip(hq.tv_show_id.head(8), hq.freq.head(8))],
            'actual_top5_w21_30': [int(p) for p in truth[u]],
            'ap5': {k: float(v) for k, v in ap.loc[u].items()},
            # places of the five that the method filled itself, before the popularity fallback (the hybrid always has
            # at least 60 popular candidates of its own)
            'filled_by_method': {name: int(min(5, ((sc.user_id == u) & (sc.score > 0) & sc.tv_show_id.isin(s.pool)).sum()))
                                 for name, sc in score_tables.items()},
            's21_lists': {name: [[int(p), seen(p), bool(p in set(truth[u]))] for p in lst[u]] for name, lst in lists.items()},
            's31_lists': {name: [int(p) for p in sub[u]] for name, sub in subs.items()}}


examples = {'A': example(user_a), 'B': example(user_b)}
ids = {p for e in examples.values() for p in [x[0] for x in e['history_top']] + e['actual_top5_w21_30']
       + [x[0] for lst in e['s21_lists'].values() for x in lst] + [p for lst in e['s31_lists'].values() for p in lst]}
examples['programmes'] = {int(p): {'title': prog.title.get(p), 'category': prog.category.get(p)} for p in sorted(ids)}
with open(os.path.join(ROOT, 'modelling', 'analysis_results.json'), 'w', encoding='utf-8') as f:
    json.dump({'map5': ap.mean().to_dict(), 'bootstrap': rows, 'segments': seg.to_dict(), 'properties': props_df.to_dict(),
               'user_a': int(user_a), 'user_b': int(user_b), 'examples': examples}, f, indent=1, default=str, ensure_ascii=False)
print(f'total runtime {time.time() - T0:.0f}s')

Submitted five programmes for user A (3525695269081370558), weeks 31 to 42


tv_show_id                                                        title category
method                      rank                                                                                  
Popularity                  1        2400480                                               'ТСН. Неделя'.     Инфо
                            2         240081                                                 Факты. День.     Инфо
                            3        2400508                             Сегодня. Итоги с Олегом Панютой.     Инфо
                            4        2400467                                             'Викна-Новости'.     Инфо
                            5       12001682                            М/с 'Губка Боб Квадратные Штаны'.    Детям
Habit x schedule            1       12001682                            М/с 'Губка Боб Квадратные Штаны'.    Детям
                            2       12002856                                     М/с 'Элвин и бурундуки'.    Детям
                            3       12001734  М/с 'Волшебные покровители', 154 с. 'Хочу это вырезать /...    Детям
                            4        2400508                             Сегодня. Итоги с Олегом Панютой.     Инфо
                            5        2400480                                               'ТСН. Неделя'.     Инфо
ALS                         1        2400467                                             'Викна-Новости'.     Инфо
                            2        2400480                                               'ТСН. Неделя'.     Инфо
                            3       12001682                            М/с 'Губка Боб Квадратные Штаны'.    Детям
                            4         240081                                                 Факты. День.     Инфо
                            5        2400508                             Сегодня. Итоги с Олегом Панютой.     Инфо
PPR                         1       12001682                            М/с 'Губка Боб Квадратные Штаны'.    Детям
                            2       12001734  М/с 'Волшебные покровители', 154 с. 'Хочу это вырезать /...    Детям
                            3       12002856                                     М/с 'Элвин и бурундуки'.    Детям
                            4        2400508                             Сегодня. Итоги с Олегом Панютой.     Инфо
                            5        2400480                                               'ТСН. Неделя'.     Инфо
Association rules           1         240081                                                 Факты. День.     Инфо
                            2        2400480                                               'ТСН. Неделя'.     Инфо
                            3        2400508                             Сегодня. Итоги с Олегом Панютой.     Инфо
                            4        2400467                                             'Викна-Новости'.     Инфо
                            5       12001682                            М/с 'Губка Боб Квадратные Штаны'.    Детям
Content-based               1        5900319                                                       Абзац!     Инфо
                            2       12002720                                             М/с 'Соник Бум'.    Детям
                            3       12002904                                    М/с 'Рыцари Нексо Найтс'.    Детям
                            4      120079707                                        М/ф 'Король обезьян'.    Детям
                            5      120080185                       М/ф 'Приключения красного самолетика'.    Детям
Hybrid (LightGBM re-ranker) 1       12001682                            М/с 'Губка Боб Квадратные Штаны'.    Детям
                            2       12001734  М/с 'Волшебные покровители', 154 с. 'Хочу это вырезать /...    Детям
                            3        2400508                             Сегодня. Итоги с Олегом Панютой.     Инфо
         

Submitted five programmes for user B (14049050536217794294), weeks 31 to 42


tv_show_id                                                  title        category
method                      rank                                                                                   
Popularity                  1        2400480                                         'ТСН. Неделя'.            Инфо
                            2         240081                                           Факты. День.            Инфо
                            3        2400508                       Сегодня. Итоги с Олегом Панютой.            Инфо
                            4        2400467                                       'Викна-Новости'.            Инфо
                            5       12001682                      М/с 'Губка Боб Квадратные Штаны'.           Детям
Habit x schedule            1        2500413                                   Гражданская оборона.            Инфо
                            2       12002793                                         М/с 'Фиксики'.           Детям
                            3        6500479                      'Мир наизнанку. Бразилия', 36 эп.  Познавательное
                            4        2400467                                       'Викна-Новости'.            Инфо
                            5        1000381       Т/с 'CSI: Место преступления', 7 сезон, 6 и 7 с.         Сериалы
ALS                         1         240081                                           Факты. День.            Инфо
                            2        2400467                                       'Викна-Новости'.            Инфо
                            3        2400480                                         'ТСН. Неделя'.            Инфо
                            4        1100411         Премьера. 'Вещдок. Особый случай. Око за око'.  Познавательное
                            5         500331                                        'Один за всех'.     Развлечения
PPR                         1        1000381       Т/с 'CSI: Место преступления', 7 сезон, 6 и 7 с.         Сериалы
                            2        6500479                      'Мир наизнанку. Бразилия', 36 эп.  Познавательное
                            3        2500413                                   Гражданская оборона.            Инфо
                            4       90057266                   Х/ф 'Терминатор 3: Восстание машин'.          Фильмы
                            5        6700512                                          Громкое дело.          Другое
Association rules           1         240081                                           Факты. День.            Инфо
                            2        2400480                                         'ТСН. Неделя'.            Инфо
                            3        2400508                       Сегодня. Итоги с Олегом Панютой.            Инфо
                            4        2400467                                       'Викна-Новости'.            Инфо
                            5       12001682                      М/с 'Губка Боб Квадратные Штаны'.           Детям
Content-based               1       10002027                                Т/с 'Дело Дойлов', 1 с.         Сериалы
                            2       10001336      Т/с 'Кости', 2 сезон, 2 с. 'Мать и дитя в бухте'.         Сериалы
                            3       10002007  Т/с 'Виктория-победительница', 1 сезон, 1 с. 'Пилот'.         Сериалы
                            4       10002197                                 Т/с 'Держись, Чарли!'.         Сериалы
                            5       10002677                             Т/с 'Дочки-матери', 1-7 с.         Сериалы
Hybrid (LightGBM re-ranker) 1        2400480                                         'ТСН. Неделя'.            Инфо
                            2        2400508                       Сегодня. Итоги с Олегом Панютой.            Инфо
                            3         500331                                        'Один 

total runtime 15s
